<a href="https://colab.research.google.com/github/JaimeB252019/PARCIAL4-BELLOSO-PALACIOS-JAIME-DANEL2520192019/blob/main/Ejer01EV04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTAR LIBRERIAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Ocultar warnings
warnings.filterwarnings("ignore")

1.
CARGA DE ARCHIVO

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/JaimeB252019/PARCIAL4-BELLOSO-PALACIOS-JAIME-DANEL2520192019/refs/heads/main/Archivos/clave_C_asociacion.csv', delimiter=',')

print("Dataset cargado correctamente")

Dataset cargado correctamente


2. MOSTRARA PRIMERAS FILAS DEL DATA SET

In [ ]:
display(df.head(10))

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,C-T0001,C-C0094,2026-01-01,Redes,Router,1,Web
1,C-T0002,C-C0001,2026-01-15,Computo,Laptop,2,Tienda
2,C-T0002,C-C0001,2026-01-15,Computo,Mouse,1,Tienda
3,C-T0003,C-C0067,2026-03-15,Redes,Adaptador,1,Web
4,C-T0003,C-C0067,2026-03-15,Accesorios,Disco_externo,2,Web
5,C-T0003,C-C0067,2026-03-15,Computo,Laptop,1,Web
6,C-T0003,C-C0067,2026-03-15,Redes,Router,1,Web
7,C-T0004,C-C0075,2026-02-11,Software,Licencia_cloud,1,Telefono
8,C-T0004,C-C0075,2026-02-11,Accesorios,Mochila,1,Telefono
9,C-T0005,C-C0011,2026-03-06,Accesorios,Disco_externo,2,Tienda


EL DATA SET CUENTA CON 7 COLUMNAS Y TIENE 601 FILAS

3. VERIFICAR VALORES NULOS DUPLICADO Y TIPOS DE DATOS

In [ ]:
print("\nVALORES NULOS")
print(df.isnull().sum())

print("\nREGISTROS DUPLICADOS")
print(df.duplicated().sum())

print("\nTIPOS DE DATOS")
print(df.dtypes)


VALORES NULOS
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

REGISTROS DUPLICADOS
1

TIPOS DE DATOS
transaccion_id    object
cliente_id        object
fecha             object
categoria         object
item              object
cantidad           int64
canal             object
dtype: object


4. PREPARAMOS DATOS EN FORMATO ADECUADO

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Preparar los datos: Crear la cesta (One-Hot Encoding)
# Agrupamos por transaccion e item para ver qué productos hay en cada compra
basket_clean = (df.groupby(['transaccion_id', 'item'])['cantidad']
          .sum().unstack().reset_index().fillna(0)
          .set_index('transaccion_id'))

# 2. Convertir a formato booleano (True si el producto está en la compra, False si no)
basket_encoded = basket_clean.applymap(lambda x: True if x > 0 else False)


5. APLICAR EL ALGORITMO APRIORI O TECNICA EQUIVALENTE

In [ ]:
frequent_items = apriori(
    basket_encoded,
    min_support=0.05,
    use_colnames=True
)

print("ITEMSETS FRECUENTES")
display(frequent_items.head())

ITEMSETS FRECUENTES


,support,itemsets
0,0.142857,(Adaptador)
1,0.205714,(Antivirus)
2,0.148571,(Audifonos)
3,0.251429,(Backup)
4,0.182857,(Cable_red)


6. GENERAR REGLAS DE ASOCIACION USANDO SOPORTE, CONFIANZA Y LIFT

In [ ]:
rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.6
)

# Ordenar por lift
rules = rules.sort_values(by="lift", ascending=False)

7. MOSTRAR LAS 10 REGLAS MAS RELEVANTES

In [ ]:
top_rules = rules[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]].head(10)

print(top_rules)

             antecedents  consequents   support  confidence      lift
3    (Cargador, Celular)  (Protector)  0.091429    0.842105  4.912281
2  (Cargador, Protector)    (Celular)  0.091429    0.800000  3.888889
4   (Protector, Celular)   (Cargador)  0.091429    0.888889  3.888889
6      (Mochila, Laptop)      (Mouse)  0.114286    0.800000  3.783784
7        (Mouse, Laptop)    (Mochila)  0.114286    0.909091  3.384913
5       (Mochila, Mouse)     (Laptop)  0.114286    0.909091  3.314394
0            (Protector)   (Cargador)  0.114286    0.666667  2.916667
1            (Protector)    (Celular)  0.102857    0.600000  2.916667


8. INTERPRETAR AL MENOS 5 REGLAS CON LENGUAJE DE NEGOCIO

In [ ]:
for i, row in top_rules.head(5).iterrows():

    antecedente = list(row['antecedents'])
    consecuente = list(row['consequents'])

    soporte = row['support']
    confianza = row['confidence']
    lift = row['lift']

    print(f"REGLA {i+1}")
    print("-----------------------------------")

    print(f"Si un cliente compra {antecedente},")
    print(f"también podría comprar {consecuente}.")

    print(f"\nSoporte: {soporte:.2f}")
    print(f"Confianza: {confianza:.2f}")
    print(f"Lift: {lift:.2f}")

    if lift > 1:
        print("Interpretación: Existe una relación positiva entre los productos.")
    else:
        print("Interpretación: La relación entre productos es débil.")

    print("\n")


REGLA 4
-----------------------------------
Si un cliente compra ['Cargador', 'Celular'],
también podría comprar ['Protector'].

Soporte: 0.09
Confianza: 0.84
Lift: 4.91
Interpretación: Existe una relación positiva entre los productos.


REGLA 3
-----------------------------------
Si un cliente compra ['Cargador', 'Protector'],
también podría comprar ['Celular'].

Soporte: 0.09
Confianza: 0.80
Lift: 3.89
Interpretación: Existe una relación positiva entre los productos.


REGLA 5
-----------------------------------
Si un cliente compra ['Protector', 'Celular'],
también podría comprar ['Cargador'].

Soporte: 0.09
Confianza: 0.89
Lift: 3.89
Interpretación: Existe una relación positiva entre los productos.


REGLA 7
-----------------------------------
Si un cliente compra ['Mochila', 'Laptop'],
también podría comprar ['Mouse'].

Soporte: 0.11
Confianza: 0.80
Lift: 3.78
Interpretación: Existe una relación positiva entre los productos.


REGLA 8
-----------------------------------
Si un clie

9. PROPONER AL MENOS 3 RECOMENDACIONES COMERCIALES BASADAS EN LOS RESULTADOS

In [ ]:
# Recomendación 1
print("1. Crear promociones combinadas")
print("   Productos que aparecen frecuentemente juntos")
print("   pueden venderse en paquetes o combos para")
print("   aumentar las ventas.\n")

# Recomendación 2
print("2. Mejorar estrategias de venta cruzada")
print("   Si un cliente compra un producto, el sistema")
print("   puede recomendar automáticamente otro producto")
print("   relacionado durante la compra.\n")

# Recomendación 3
print("3. Optimizar ubicación de productos")
print("   Los productos asociados pueden colocarse")
print("   cerca en la tienda física o mostrarse juntos")
print("   en plataformas digitales.\n")

1. Crear promociones combinadas
   Productos que aparecen frecuentemente juntos
   pueden venderse en paquetes o combos para
   aumentar las ventas.

2. Mejorar estrategias de venta cruzada
   Si un cliente compra un producto, el sistema
   puede recomendar automáticamente otro producto
   relacionado durante la compra.

3. Optimizar ubicación de productos
   Los productos asociados pueden colocarse
   cerca en la tienda física o mostrarse juntos
   en plataformas digitales.



En el desarrollo del análisis se cargó y revisó el archivo CSV para verificar que los datos estuvieran completos y correctamente estructurados. Posteriormente, se validaron valores nulos, registros duplicados y tipos de datos para asegurar la calidad de la información antes de aplicar el modelo.

Después, los datos fueron preparados en formato binario para poder utilizar el algoritmo Apriori, el cual permitió identificar patrones de compra frecuentes y relaciones entre productos mediante reglas de asociación. Estas reglas fueron evaluadas utilizando métricas como soporte, confianza y lift, permitiendo determinar cuáles asociaciones eran más relevantes dentro del conjunto de datos.

Finalmente, con base en los resultados obtenidos, se interpretaron las reglas más importantes desde una perspectiva comercial, generando recomendaciones orientadas a mejorar promociones, ventas cruzadas y organización de productos para apoyar la toma de decisiones del negocio.